## Load in data

In [32]:
import pandas as pd

In [34]:
dataset = pd.read_parquet('../data/processed/features.parquet')
dataset.head()

,user_id,item_id,rating,timestamp,movie_title,release_date,video_release_date,imdb_url,genre_0,genre_1,...,genre_14,genre_15,genre_16,genre_17,genre_18,age,gender,occupation,zip_code,click
0,196,242,3,881250949,Kolya (1996),24-Jan-1997,NaN,http://us.imdb.com/M/title-exact?Kolya%20(1996),0,0,...,0,0,0,0,0,49,M,writer,55105,0
1,186,302,3,891717742,L.A. Confidential (1997),01-Jan-1997,NaN,http://us.imdb.com/M/title-exact?L%2EA%2E+Conf...,0,0,...,0,0,1,0,0,39,F,executive,00000,0
2,22,377,1,878887116,Heavyweights (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?Heavyweights%...,0,0,...,0,0,0,0,0,25,M,writer,40206,0
3,244,51,2,880606923,Legends of the Fall (1994),01-Jan-1994,NaN,http://us.imdb.com/M/title-exact?Legends%20of%...,0,0,...,1,0,0,1,1,28,M,technician,80525,0
4,166,346,1,886397596,Jackie Brown (1997),01-Jan-1997,NaN,http://us.imdb.com/M/title-exact?imdb-title-11...,0,0,...,0,0,0,0,0,47,M,educator,55113,0


## Train The Models

### Variant A : Collaborative Filtering


For our first model, we are going to do collaborative filtering. In collaborative filtering, we recommend items to users that other similar users have also liked.

For our first baseline model we will be using KNNwithMeans which builds upon normal KNNBasic. In KNNBasic, we use the K nearest neighbours' ratings at face values, whereas 'withMeans' adjusts for user bias in terms of how they rate by centering ratings around user averages. Additionally, we are using cosine similarity as our distance metric

In [51]:
from surprise import Dataset, Reader, KNNWithMeans

# create the reader for rating
reader = Reader(rating_scale=(1, 5))

# load the dataset from the data from the previous notebook
data = Dataset.load_from_df(dataset[['user_id', 'item_id', 'rating']], reader)

# we are going to do item-based 
sim_options = {"name" : "cosine", "user_based": False}
# use KNNWithMeans algorithm for collaborative filtering
algo = KNNWithMeans(sim_options=sim_options)

In [53]:
from surprise.model_selection import train_test_split
from surprise import accuracy

trainset, testset = train_test_split(data, test_size= 0.2)

# train the algorithm on the train set
algo.fit(trainset)

# get the predictions of the test set
predictions = algo.test(testset)

Computing the cosine similarity matrix...
Done computing similarity matrix.


Now that we have the predictions we should calculate the root mean square error and mean absolute error. We are also converting the predicted ratings into a probability of that movie being clicked by the user; from this we can get the simulated CTR for variant A. We also calculate AUC which is the area under the curve indicating how well our model distinguished between classes (click / no click), getting a score of 0.774 (where 1 perfect).

In [58]:
print(f"RMSE: {accuracy.rmse(predictions)}")
print(f"MAE: {accuracy.mae(predictions)}")

RMSE: 0.9413
RMSE: 0.9412882576111511
MAE:  0.7386
MAE: 0.7386044554402571


In [64]:
# get the predicted ratings 
pred_ratings = [pred.est for pred in predictions]

# get the true ratings
true_ratings = [pred.r_ui for pred in predictions]

# comvert the probably of a person clicking on a movie according to the predicted rating
pred_click_prob = [(r - 1) / 4 for r in pred_ratings]

In [72]:
from sklearn.metrics import roc_auc_score

# calculate the average CTR for our first model by taking average of all probabilities
variant_a_ctr = sum(pred_click_prob) / len(pred_click_prob)
print(f"Variant A simulated CTR: {variant_a_ctr:.4f}")

# get the true clicks based on 5 star threshold
true_clicks = [1 if r >= 5 else 0 for r in true_ratings] 
# get the auc
auc = roc_auc_score(true_clicks, pred_click_prob)
print(f"Variant A AUC: {auc:.4f}")


Variant A simulated CTR: 0.6353
Variant A AUC: 0.7774
